# Building Observable Grok Agents

When you chain multiple Grok API calls together—an agent deciding which tools to call, executing them, and looping until it has an answer—things get complex fast. Which tools did the agent pick? Why did it call the same function twice? Where's the latency bottleneck? How many tokens did the whole interaction cost?

This guide builds a practical Grok-powered travel planning agent with [function calling](https://docs.x.ai/developers/tools/function-calling), then adds observability using [MLflow Tracing](https://mlflow.org/docs/latest/genai/tracing/) so you can inspect every decision, tool call, and response in a visual trace viewer.

## Table of Contents
- [Setup](#setup)
- [Define the Agent's Tools](#define-the-agents-tools)
- [Build the Agent Loop](#build-the-agent-loop)
- [Run the Agent](#run-the-agent)
- [Analyzing Traces](#analyzing-traces)
- [Conclusion](#conclusion)

## Setup

In [1]:
%pip install --quiet openai mlflow python-dotenv

Note: you may need to restart the kernel to use updated packages.


> **Note:** Make sure to export an env var named `XAI_API_KEY` or set it in a `.env` file at the root of this repo if you want to run the notebook. Head over to our [console](https://console.x.ai/) to obtain an api key if you don't have one already.

In [2]:
import os

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

XAI_API_KEY = os.getenv("XAI_API_KEY")
if not XAI_API_KEY:
    raise ValueError("XAI_API_KEY is not set")

client = OpenAI(
    base_url="https://api.x.ai/v1",
    api_key=XAI_API_KEY,
)

### Enable Observability

Since xAI's Grok API is [OpenAI-compatible](https://docs.x.ai/docs/guides/migration#from-openai-to-xai), we can add automatic tracing with [MLflow](https://mlflow.org/) in two lines. Every Grok API call will be recorded—including inputs, outputs, latency, and token usage—with zero changes to our application code.

Start the MLflow tracking server in a separate terminal:

```bash
mlflow server --port 5000
```

Then connect to it and enable auto-tracing:

In [3]:
import mlflow

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("Grok-Travel-Agent")
mlflow.openai.autolog()

## Define the Agent's Tools

Our travel planning agent can call four tools to gather information for the user:

| Tool | Purpose |
|------|---------|
| `search_destinations` | Find destinations matching user preferences |
| `get_flight_info` | Look up flight prices and times |
| `get_weather` | Check weather forecast for a city |
| `convert_currency` | Convert between currencies |

In a production application, these would call real APIs. Here we use mock implementations to keep the notebook self-contained.

In [4]:
import json

# Tool definitions in OpenAI function-calling format
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "search_destinations",
            "description": "Search for travel destinations based on preferences like climate, budget, and activities.",
            "parameters": {
                "type": "object",
                "properties": {
                    "preferences": {
                        "type": "string",
                        "description": "User's travel preferences, e.g. 'warm beach destination under $1000'",
                    }
                },
                "required": ["preferences"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_flight_info",
            "description": "Get flight prices and duration between two cities.",
            "parameters": {
                "type": "object",
                "properties": {
                    "origin": {"type": "string", "description": "Departure city"},
                    "destination": {"type": "string", "description": "Arrival city"},
                },
                "required": ["origin", "destination"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather forecast for a city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "City name"},
                },
                "required": ["city"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "convert_currency",
            "description": "Convert an amount between currencies.",
            "parameters": {
                "type": "object",
                "properties": {
                    "amount": {"type": "number", "description": "Amount to convert"},
                    "from_currency": {"type": "string", "description": "Source currency code (e.g. USD)"},
                    "to_currency": {"type": "string", "description": "Target currency code (e.g. EUR)"},
                },
                "required": ["amount", "from_currency", "to_currency"],
            },
        },
    },
]

In [5]:
# Mock tool implementations
MOCK_DATA = {
    "destinations": [
        {"city": "Cancún", "country": "Mexico", "highlights": "Beaches, Mayan ruins, nightlife", "avg_daily_cost": "$120"},
        {"city": "San Juan", "country": "Puerto Rico", "highlights": "Old San Juan, beaches, no passport needed (US)", "avg_daily_cost": "$150"},
        {"city": "Medellín", "country": "Colombia", "highlights": "Spring-like weather, culture, affordable", "avg_daily_cost": "$70"},
    ],
    "flights": {
        ("San Francisco", "Cancún"): {"price": "$380", "duration": "5h 30m", "airline": "United"},
        ("San Francisco", "San Juan"): {"price": "$420", "duration": "7h 15m", "airline": "JetBlue"},
        ("San Francisco", "Medellín"): {"price": "$350", "duration": "8h 45m", "airline": "Avianca"},
    },
    "weather": {
        "Cancún": {"temperature": "88°F / 31°C", "condition": "Sunny with afternoon clouds", "humidity": "75%"},
        "San Juan": {"temperature": "84°F / 29°C", "condition": "Partly cloudy, chance of brief showers", "humidity": "80%"},
        "Medellín": {"temperature": "77°F / 25°C", "condition": "Pleasant, light rain in the evening", "humidity": "65%"},
    },
    "exchange_rates": {"USD_MXN": 17.15, "USD_COP": 4150.0, "USD_EUR": 0.92, "USD_GBP": 0.79},
}


@mlflow.trace(span_type="TOOL")
def search_destinations(preferences: str) -> str:
    return json.dumps(MOCK_DATA["destinations"], indent=2)


@mlflow.trace(span_type="TOOL")
def get_flight_info(origin: str, destination: str) -> str:
    # Fuzzy match on city names
    for (orig, dest), info in MOCK_DATA["flights"].items():
        if orig.lower() in origin.lower() and dest.lower() in destination.lower():
            return json.dumps({"origin": orig, "destination": dest, **info})
    return json.dumps({"error": f"No flights found from {origin} to {destination}"})


@mlflow.trace(span_type="TOOL")
def get_weather(city: str) -> str:
    for name, info in MOCK_DATA["weather"].items():
        if name.lower() in city.lower():
            return json.dumps({"city": name, **info})
    return json.dumps({"error": f"Weather data not available for {city}"})


@mlflow.trace(span_type="TOOL")
def convert_currency(amount: float, from_currency: str, to_currency: str) -> str:
    key = f"{from_currency.upper()}_{to_currency.upper()}"
    rate = MOCK_DATA["exchange_rates"].get(key)
    if rate:
        converted = round(amount * rate, 2)
        return json.dumps({"amount": amount, "from": from_currency, "to": to_currency, "converted": converted, "rate": rate})
    return json.dumps({"error": f"Exchange rate not available for {from_currency} to {to_currency}"})


TOOL_FUNCTIONS = {
    "search_destinations": search_destinations,
    "get_flight_info": get_flight_info,
    "get_weather": get_weather,
    "convert_currency": convert_currency,
}

## Build the Agent Loop

The core of any tool-calling agent is a loop: send a message to the model, check if it wants to call tools, execute them, feed the results back, and repeat until the model produces a final text response.

We wrap the entire loop with `@mlflow.trace` so that all the API calls and tool executions within a single user query are grouped into one trace.

In [6]:
SYSTEM_PROMPT = """You are a helpful travel planning assistant. Use the available tools to research 
destinations, check flights, weather, and currency conversions to help users plan their trips. 
Always gather concrete data before making recommendations. Be concise in your final response."""


@mlflow.trace(span_type="AGENT")
def run_agent(user_message: str, max_iterations: int = 10) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_message},
    ]

    for i in range(max_iterations):
        response = client.chat.completions.create(
            model="grok-4-latest",
            messages=messages,
            tools=TOOLS,
        )
        choice = response.choices[0]

        # If the model produced a final text response, we're done
        if choice.finish_reason == "stop":
            return choice.message.content

        # Otherwise, execute each tool call and feed results back
        messages.append(choice.message)

        for tool_call in choice.message.tool_calls:
            func = TOOL_FUNCTIONS[tool_call.function.name]
            args = json.loads(tool_call.function.arguments)
            result = func(**args)

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result,
            })

    return "Agent reached maximum iterations without a final response."

## Run the Agent

Let's give the agent a real planning task. This will trigger multiple tool calls as the agent researches destinations, checks flights, and looks up weather—all automatically traced.

In [7]:
result = run_agent(
    "I want to plan a weekend getaway from San Francisco. "
    "I'm looking for somewhere warm with beaches, and my budget is around $800 total. "
    "What are my best options?"
)
print(result)

Based on your preferences for a warm beach weekend getaway from San Francisco within ~$800 total budget (assuming 2-3 days, including round-trip flights, lodging, food, and activities), here are the best options from my research. I prioritized destinations with beaches and current warm weather (80°F+). Costs are estimates; actuals vary by dates/bookings.

### Top Recommendation: Cancún, Mexico
- **Why it fits**: Stunning beaches, Mayan ruins, and vibrant nightlife. Current weather: 88°F (31°C), sunny with some clouds.
- **Flights**: Round-trip from San Francisco ~$380, 5.5 hours each way (e.g., via United).
- **Estimated total cost**: $700–$800 (flights + ~$120/day for 3 days covering budget lodging, meals, and local transport).
- **Tips**: Passport required. Great for relaxation or water activities. Book soon for deals.

### Alternative: San Juan, Puerto Rico
- **Why it fits**: Beautiful beaches, historic Old San Juan, and easy access (no passport needed for US citizens). Current weat

Trace(trace_id=tr-8fb12c3077976aefff15156ddf901c13)

## Analyzing Traces

Open the MLflow UI at [http://localhost:5000](http://localhost:5000) and navigate to the **Grok-Travel-Agent** experiment. You'll see a trace for each `run_agent` call.

### What each trace shows

Click on a trace to see the full span tree:

![MLflow trace viewer showing the travel agent's execution](images/trace_viewer.png)

- **Parent span** (`run_agent`): The entire agent interaction from user query to final answer, showing total latency and the input/output.
- **Child spans** (one per Grok API call): Each iteration of the agent loop is a separate span. You can inspect:
  - **Inputs**: The full messages array, including tool results from previous iterations
  - **Outputs**: The model's response—either tool calls or a final text answer
  - **Token usage**: Input and output tokens per call
  - **Latency**: How long each API call took
- **Tool spans**: Each tool execution (`search_destinations`, `get_flight_info`, `get_weather`, etc.) appears as its own span with inputs and outputs, so you can verify the data your agent received.

### What to look for

| Question | Where to find the answer |
|----------|-------------------------|
| Which tools did the agent call? | Look at the tool spans between the Completions spans |
| Which API call was slowest? | Compare latency across child spans in the timeline view |
| How many tokens did this cost? | Sum the token usage across all child spans |
| Did the agent get the right context? | Inspect the messages list in LLM span's input to see exactly what context was sent |

## Conclusion

In this guide, we built a Grok-powered travel agent with function calling and added observability to trace its behavior. Key takeaways:

- **Agents are multi-step**: A single user query can trigger many API calls and tool executions. Observability turns this opaque loop into an inspectable sequence.
- **Tracing is low-overhead**: `mlflow.openai.autolog()` instruments all Grok API calls automatically. `@mlflow.trace` groups related calls into a single trace.
- **Traces help you debug and optimize**: See which tools the agent picked, where latency hides, and how token costs add up.

### Next Steps

- [MLflow Tracing documentation](https://mlflow.org/docs/latest/genai/tracing/) — custom spans, trace search, and evaluation
- [MLflow xAI / Grok integration guide](https://mlflow.org/docs/latest/genai/tracing/integrations/listing/xai-grok) — detailed setup and troubleshooting
- [xAI Tools documentation](https://docs.x.ai/developers/tools/overview) — built-in tools like web search and X search